# 01 - Cargar los viajes: abrir los ZIP y armar la tabla bronze

**Antes de correr este notebook.** Databricks no se puede conectar a la página de la Intendencia. Por eso bajamos a mano el ZIP de cada mes desde el portal de datos abiertos y lo subimos al volume, en la carpeta `viajes_zip/`. El nombre del ZIP no importa: el notebook descubre el mes mirando el nombre del ZIP o el del CSV que tiene adentro.

**Parte A. Abrir los ZIP.** Saca el CSV de cada ZIP y lo guarda como `viajes/viajes_stm_AAAA_MM.csv`.

**Parte B. Tabla bronze.** Copia cada mes a la tabla `viajes_bronze` **sin limpiar nada**. Todas las columnas se guardan como texto y se agregan tres columnas para saber de dónde vino cada fila (mes, archivo y fecha de carga).

Se puede correr varias veces. Los meses que ya están hechos se saltean, salvo que se pida rehacerlos.

In [0]:
%run ./00_setup

## Opciones

Antes de correr, elegí qué hacer cambiando estos valores en la celda de abajo:

| Opción | Valor | Qué pasa |
|---|---|---|
| `MESES` | `["2024-01"]` | Procesa solo esos meses. **La primera vez, probá con un solo mes.** |
| `MESES` | `None` | Procesa todos los meses que estén subidos. |
| `FUENTE` | `"zip"` | Abre los ZIP de `viajes_zip/` (Parte A) y después carga bronze. |
| `FUENTE` | `"volume"` | Se saltea la Parte A, porque los CSV ya están en `viajes/`. |
| `FORZAR_BRONZE` | `True` | Vuelve a cargar meses que ya estaban en bronze. |

In [0]:
MESES = None             # None quiere decir todos los meses
FUENTE = "zip"           # "zip" o "volume"
CARPETA_ZIP = config.RAW_ZIP   # cambiar si subiste los ZIP a otra carpeta
FORZAR_BRONZE = False
 
assert FUENTE in ("zip", "volume"), "FUENTE debe ser 'zip' o 'volume'"
CARPETA_CSV = config.RAW_VIAJES

In [0]:
import importlib
from pathlib import Path
 
from pyspark.sql import functions as F
 
# Funciona porque antes corrimos %run ./00_setup, que le dice a Python dónde está src/
from bondis import ingesta
 
importlib.reload(ingesta)
 
 
def mes_elegido(anio_mes):
    """Verdadero si hay que procesar el mes 'AAAA-MM' según la opción MESES."""
    return MESES is None or anio_mes in MESES

## Parte A. Abrir los ZIP que bajamos a mano

Para cada ZIP de la carpeta:
1. Descubre de qué mes es, por el nombre del ZIP (o, si no se puede, por el nombre del CSV de adentro).
2. Saca el CSV y lo guarda como `raw/viajes/viajes_stm_AAAA_MM.csv`.
3. Revisa que el CSV que sacó tenga el mismo tamaño que dice el ZIP. Si no, el archivo quedó cortado.

Los ZIP no se borran. Si un CSV ya estaba sacado, se saltea.

**Mirá la tabla de meses de la próxima celda antes de seguir.** Ahí se ve si hay meses repetidos o meses que faltan.

In [0]:
if FUENTE == "zip":
    # Busca también en subcarpetas, y acepta .zip, .ZIP o .Zip
    zips = sorted(p for p in Path(CARPETA_ZIP).rglob("*") if p.is_file() and p.suffix.lower() == ".zip")
    print(f"ZIP encontrados en {CARPETA_ZIP}: {len(zips)}\n")
 
    # Primero solo mostramos qué mes le toca a cada ZIP, sin sacar nada todavía.
    # Si el nombre del ZIP no tiene el mes, se mira el nombre del CSV de adentro.
    mes_por_zip = {}
    print(f"{'ZIP':<34} {'Mes':<8} {'ZIP':>8} {'CSV':>8}  CSV interno")
    for z in sorted(zips, key=lambda p: ingesta.mes_de_zip(p) or "9999"):
        am = ingesta.mes_de_zip(z)
        mes_por_zip[z] = am
        internos = ingesta.contenido_zip(z)
        nombre_csv = internos[0][0] if internos else "(sin CSV)"
        mb_csv = sum(mb for _, mb in internos)
        print(f"{z.name:<34} {am or '¿?':<8} {z.stat().st_size / 1e6:>6,.0f}MB {mb_csv:>6,.0f}MB  {nombre_csv}")
 
    # Revisamos si hay meses repetidos o meses que faltan entre 2024 y 2025
    from collections import Counter
    conteo = Counter(am for am in mes_por_zip.values() if am)
    repetidos = sorted(am for am, n in conteo.items() if n > 1)
    esperados = (
        spark.sql(f"SELECT explode(sequence(to_date('{config.FECHA_INICIO}'), "
                  f"date_sub(to_date('{config.FECHA_FIN}'), 1), interval 1 month)) AS d")
        .selectExpr("date_format(d, 'yyyy-MM') AS am").toPandas()["am"].tolist()
    )
    faltantes = [am for am in esperados if am not in conteo]
    print(f"\nMeses identificados: {len(conteo)}  |  sin identificar: {sum(1 for a in mes_por_zip.values() if not a)}")
    print(f"Meses repetidos: {repetidos or 'ninguno'}")
    print(f"Meses del período sin ZIP: {faltantes or 'ninguno'}")

In [0]:
errores_zip = []
 
if FUENTE == "zip":
    for z in zips:
        am = ingesta.mes_desde_texto(z.stem)
        if am is not None and not mes_elegido(am):
            continue
        try:
            am, _ = ingesta.descomprimir_zip(z, config.RAW_VIAJES)
        except Exception as e:  # si un ZIP falla, se anota el error y se sigue con el próximo
            errores_zip.append((z.name, str(e)))
            print(f"[FAIL] {z.name}: {e}")
 
    print(f"\nErrores: {len(errores_zip)}")
    for nombre, err in errores_zip:
        print(f"  {nombre}: {err}")

## Parte B. Cargar a bronze
### B.1 Qué archivos hay y qué meses ya están cargados

In [0]:
archivos = sorted(Path(CARPETA_CSV).glob("viajes_stm_*.csv"))
print(f"Carpeta de origen: {CARPETA_CSV}  ({len(archivos)} CSV de viajes en total)")
por_mes = {am: ps for am, ps in ingesta.agrupar_por_mes(archivos).items() if mes_elegido(am)}
 
tabla_existe = spark.catalog.tableExists(config.TABLE_BRONZE)
cargados = set()
if tabla_existe:
    cargados = {
        r["anio_mes"]
        for r in spark.table(config.TABLE_BRONZE).select("anio_mes").distinct().collect()
    }
 
pendientes = {am: ps for am, ps in por_mes.items() if FORZAR_BRONZE or am not in cargados}
 
print(f"Meses con CSV en el volume: {len(por_mes)}")
print(f"Meses ya cargados en bronze: {len(cargados & set(por_mes))}")
print(f"Meses a cargar ahora:        {len(pendientes)}  {list(pendientes)}")

### B.2 Cargar mes por mes

Para cada mes:
1. Mira el principio del archivo para saber qué símbolo separa las columnas (coma, punto y coma...) y cómo están guardadas las tildes.
2. Lee el CSV con **todas las columnas como texto**. En bronze no decidimos si algo es número o fecha, eso se hace en silver.
3. Deja prolijos los nombres de las columnas (sin tildes ni espacios).
4. Agrega `anio_mes`, `archivo_origen` y `fecha_ingesta`, para saber de dónde salió cada fila.
5. Guarda el mes en la tabla. Si el mes ya estaba, **reemplaza solo ese mes** y deja los demás como estaban (eso hace `replaceWhere`).

In [0]:
columnas_por_mes = {}
 
for am, paths in pendientes.items():
    sep, enc, encabezado = ingesta.detectar_formato(paths[0])
    print(f"[{am}] {len(paths)} archivo/s, separador {sep!r}, {enc}")
 
    crudo = (
        spark.read
        .option("header", True)
        .option("sep", sep)
        .option("encoding", enc)
        .option("quote", '"')
        .option("escape", '"')
        .option("inferSchema", False)
        .csv(paths)
    )
 
    nuevos_nombres = ingesta.normalizar_columnas(crudo.columns)
    df = crudo.select(
        *[F.col(f"`{c}`").alias(n) for c, n in zip(crudo.columns, nuevos_nombres)],
        F.lit(am).alias("anio_mes"),
        F.col("_metadata.file_name").alias("archivo_origen"),
        F.current_timestamp().alias("fecha_ingesta"),
    )
 
    escritor = df.write.format("delta").mode("overwrite").option("mergeSchema", "true")
    if tabla_existe:
        escritor.option("replaceWhere", f"anio_mes = '{am}'").saveAsTable(config.TABLE_BRONZE)
    else:
        escritor.partitionBy("anio_mes").saveAsTable(config.TABLE_BRONZE)
        tabla_existe = True
 
    filas = spark.table(config.TABLE_BRONZE).where(F.col("anio_mes") == am).count()
    columnas_por_mes[am] = nuevos_nombres
    print(f"        {filas:,} filas, columnas: {nuevos_nombres}")

### B.3 ¿Todos los meses tienen las mismas columnas?

Si la Intendencia cambió el formato de un año a otro, algunos meses pueden traer columnas distintas. Bronze las guarda todas igual (eso hace `mergeSchema`). Si pasa, se arregla en silver.

In [0]:
if columnas_por_mes:
    esquemas = {}
    for am, cols in columnas_por_mes.items():
        esquemas.setdefault(tuple(cols), []).append(am)
    if len(esquemas) == 1:
        print("Todos los meses cargados tienen las mismas columnas.")
    else:
        print(f"ATENCIÓN: {len(esquemas)} esquemas distintos.")
        for cols, meses in esquemas.items():
            print(f"  Meses {meses}:\n    {list(cols)}")
else:
    print("No se cargaron meses en esta ejecución.")

## Revisión final

Cuántas filas y archivos quedaron por mes, y cómo se ven las primeras filas.

In [0]:
if spark.catalog.tableExists(config.TABLE_BRONZE):
    display(
        spark.table(config.TABLE_BRONZE)
        .groupBy("anio_mes")
        .agg(F.count("*").alias("filas"), F.countDistinct("archivo_origen").alias("archivos"))
        .orderBy("anio_mes")
    )

In [0]:
if spark.catalog.tableExists(config.TABLE_BRONZE):
    spark.table(config.TABLE_BRONZE).printSchema()
    display(spark.table(config.TABLE_BRONZE).limit(10))